In [1]:
import os
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle API token configured successfully.")

Kaggle API token configured successfully.


In [2]:
import os

dataset_path = "flickr8k_dataset"

if not os.path.exists(dataset_path):
    print("Dataset not found. Downloading...")

    !kaggle datasets download -d adityajn105/flickr8k
    !unzip -q flickr8k.zip -d flickr8k_dataset

    print("Dataset downloaded and extracted to the 'flickr8k_dataset' folder.")
else:
    print("Dataset already exists. Skipping download.")

Dataset not found. Downloading...
Dataset URL: https://www.kaggle.com/datasets/adityajn105/flickr30k
License(s): CC0-1.0
100% 8.16G/8.16G [01:24<00:00, 103MB/s]

Dataset downloaded and extracted to the 'flickr30k_dataset' folder.


In [3]:
import numpy as np
import pandas as pd

dataset_path = 'flickr8k_dataset'
images_folder = os.path.join(dataset_path, 'Images')
captions_file = os.path.join(dataset_path, 'captions.txt')

df = pd.read_csv(captions_file)

print(f"Total caption pairs found: {len(df)}")
print("\n--- Sample Data ---")
print(df.head())

unique_images = df['image'].unique()
print(f"\nTotal unique images: {len(unique_images)}")

SEED = 42
val_size = 1000
rng = np.random.default_rng(SEED)
shuffled_images = rng.permutation(unique_images)

val_image_names = shuffled_images[:val_size]
train_image_names = shuffled_images[val_size:]

train_df = df[df['image'].isin(train_image_names)].reset_index(drop=True)
val_df = df[df['image'].isin(val_image_names)].reset_index(drop=True)

print(f"\nTraining set: {len(train_df)} captions (from {len(train_image_names)} images)")
print(f"Validation set: {len(val_df)} captions (from {len(val_image_names)} images)")
print(f"Random seed used for split: {SEED}")

Total caption pairs found: 158915

--- Sample Data ---
            image                                            caption
0  1000092795.jpg   Two young guys with shaggy hair look at their...
1  1000092795.jpg   Two young , White males are outside near many...
2  1000092795.jpg   Two men in green shirts are standing in a yard .
3  1000092795.jpg       A man in a blue shirt standing in a garden .
4  1000092795.jpg            Two friends enjoy time spent together .

Total unique images: 31783

Training set: 153915 captions (from 30783 images)
Validation set: 5000 captions (from 1000 images)
Random seed used for split: 42


In [4]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from collections import Counter

class Vocabulary:
    def __init__(self, freq_threshold):
        self.freq_threshold = freq_threshold

        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.idx = 4

    def __len__(self):
        return len(self.itos)

    def tokenize(self, text):
        return nltk.tokenize.word_tokenize(text.lower())

    def build_vocabulary(self, sentence_list):
        frequencies = Counter()

        for sentence in sentence_list:
            for word in self.tokenize(sentence):
                frequencies[word] += 1

        for word, count in frequencies.items():
            if count >= self.freq_threshold:
                self.stoi[word] = self.idx
                self.itos[self.idx] = word
                self.idx += 1

    def numericalize(self, text):
        tokenized_text = self.tokenize(text)
        return [self.stoi.get(word, self.stoi["<UNK>"]) for word in tokenized_text]

print("Vocabulary class defined successfully.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Vocabulary class defined successfully.


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [5]:
from PIL import Image
import torch
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

class FlickrDataset(Dataset):
    def __init__(self, root_dir, df, transform=None, vocab=None, freq_threshold=5):
        self.root_dir = root_dir
        self.df = df
        self.transform = transform

        self.imgs = self.df["image"]
        self.captions = self.df["caption"]

        if vocab is None:
            self.vocab = Vocabulary(freq_threshold)
            self.vocab.build_vocabulary(self.captions.tolist())
        else:
            self.vocab = vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        caption = self.captions[index]
        img_id = self.imgs[index]

        img_path = os.path.join(self.root_dir, img_id)
        img = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        numericalized_caption = [self.vocab.stoi["<SOS>"]]
        numericalized_caption += self.vocab.numericalize(caption)
        numericalized_caption.append(self.vocab.stoi["<EOS>"])

        return img, torch.tensor(numericalized_caption)

class CapsCollate:
    def __init__(self, pad_idx):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        imgs = [item[0].unsqueeze(0) for item in batch]
        imgs = torch.cat(imgs, dim=0)

        targets = [item[1] for item in batch]

        targets = pad_sequence(targets, batch_first=True, padding_value=self.pad_idx)

        return imgs, targets

print("Dataset and Collate functions defined successfully.")

Dataset and Collate functions defined successfully.


In [6]:
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    normalize,
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize,
])

dataset_path = 'flickr8k_dataset'
images_folder = os.path.join(dataset_path, 'Images')

# Ensure captions are strings and handle NaN values before creating the dataset
train_df['caption'] = train_df['caption'].astype(str).fillna('')
val_df['caption'] = val_df['caption'].astype(str).fillna('')

train_dataset = FlickrDataset(
    root_dir=images_folder,
    df=train_df,
    transform=train_transform
)

val_dataset = FlickrDataset(
    root_dir=images_folder,
    df=val_df,
    transform=val_transform,
    vocab=train_dataset.vocab
)

batch_size = 64
pad_idx = train_dataset.vocab.stoi["<PAD>"]

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    shuffle=True,
    collate_fn=CapsCollate(pad_idx=pad_idx)
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    shuffle=False,
    collate_fn=CapsCollate(pad_idx=pad_idx)
)

print(f"Total vocabulary size: {len(train_dataset.vocab)}")
print(f"Number of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")

Total vocabulary size: 7644
Number of training batches: 2405
Number of validation batches: 79


In [7]:
import torch
import torch.nn as nn
import torchvision.models as models
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using hardware: {device}")

class EncoderCNN(nn.Module):
    def __init__(self, embed_size):
        super(EncoderCNN, self).__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.resnet = nn.Sequential(*list(resnet.children())[:-2])
        self.conv = nn.Conv2d(2048, embed_size, kernel_size=1)
        self.bn = nn.BatchNorm2d(embed_size, momentum=0.01)

    def forward(self, images):
        features = self.resnet(images)
        features = self.conv(features)
        features = self.bn(features)
        features = features.flatten(2).permute(0, 2, 1)
        return features

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class DecoderTransformer(nn.Module):
    def __init__(self, embed_size, vocab_size, num_heads=8, num_layers=3, dropout=0.3):
        super(DecoderTransformer, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.pos_encoder = PositionalEncoding(embed_size)
        
        decoder_layer = nn.TransformerDecoderLayer(d_model=embed_size, nhead=num_heads, dropout=dropout, batch_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        
        self.linear = nn.Linear(embed_size, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, features, captions):
        captions_input = captions[:, :-1]
        seq_length = captions_input.size(1)
        
        embeddings = self.dropout(self.pos_encoder(self.embed(captions_input)))
        tgt_mask = self.generate_square_subsequent_mask(seq_length).to(features.device)
        tgt_key_padding_mask = (captions_input == 0)
        
        outputs = self.transformer_decoder(
            tgt=embeddings,
            memory=features,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        )
        outputs = self.linear(outputs)
        return outputs

Using hardware: cuda


In [8]:
import torch.optim as optim
import torch.nn as nn

embed_size = 256
vocab_size = len(train_dataset.vocab)
num_heads = 8
num_layers = 3
decoder_dropout = 0.3
decoder_learning_rate = 3e-4
encoder_learning_rate = 1e-4
weight_decay = 1e-5
num_epochs = 20
early_stopping_patience = 4

encoder = EncoderCNN(embed_size).to(device)
decoder = DecoderTransformer(embed_size, vocab_size, num_heads=num_heads, num_layers=num_layers, dropout=decoder_dropout).to(device)

pad_idx = train_dataset.vocab.stoi["<PAD>"]
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

for param in encoder.resnet[-1].parameters():
    param.requires_grad = True

decoder_params = list(decoder.parameters())
encoder_layer4_params = list(encoder.resnet[-1].parameters())
encoder_conv_params = list(encoder.conv.parameters())
encoder_bn_params = list(encoder.bn.parameters())
trainable_params = decoder_params + encoder_layer4_params + encoder_conv_params + encoder_bn_params

optimizer = optim.AdamW([
    {"params": decoder_params, "lr": decoder_learning_rate},
    {"params": encoder_layer4_params, "lr": encoder_learning_rate},
    {"params": encoder_conv_params, "lr": decoder_learning_rate},
    {"params": encoder_bn_params, "lr": decoder_learning_rate},
], weight_decay=weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

print(f"Models initialized and moved to {device}.")
print(f"Vocabulary Size: {vocab_size}")
print("Fine-tuning: decoder + encoder.layer4 + encoder.conv")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 180MB/s]


Models initialized and moved to cuda.
Vocabulary Size: 7644
Fine-tuning: decoder + encoder.layer4 + encoder.fc


In [9]:
from tqdm import tqdm
import pickle

print("Starting Training Engine...")

best_val_loss = float('inf')
epochs_without_improvement = 0

for epoch in range(num_epochs):
    print(f"\n=== Epoch {epoch+1}/{num_epochs} ===")

    # ==========================
    #       TRAINING PHASE
    # ==========================
    encoder.train()
    decoder.train()
    train_loss = 0.0

    loop = tqdm(train_loader, total=len(train_loader), desc="Training")

    for idx, (imgs, captions) in enumerate(loop):
        imgs = imgs.to(device)
        captions = captions.to(device)

        optimizer.zero_grad()

        features = encoder(imgs)
        outputs = decoder(features, captions)

        loss = criterion(outputs.reshape(-1, vocab_size), captions[:, 1:].reshape(-1))

        loss.backward()

        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / len(train_loader)

    # ==========================
    #      VALIDATION PHASE
    # ==========================
    encoder.eval()
    decoder.eval()
    val_loss = 0.0

    with torch.no_grad():
        for imgs, captions in val_loader:
            imgs = imgs.to(device)
            captions = captions.to(device)

            features = encoder(imgs)
            outputs = decoder(features, captions)

            loss = criterion(outputs.reshape(-1, vocab_size), captions[:, 1:].reshape(-1))
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)
    scheduler.step(avg_val_loss)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1} Completed | Train Loss: {avg_train_loss:.4f} | Validation Loss: {avg_val_loss:.4f} | Decoder LR: {current_lr:.6f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_without_improvement = 0
        torch.save(encoder.state_dict(), 'encoder_weights.pth')
        torch.save(decoder.state_dict(), 'decoder_weights.pth')
        with open('vocab.pkl', 'wb') as f:
            pickle.dump(train_dataset.vocab, f)
        print('Validation improved. Best checkpoint saved.')
    else:
        epochs_without_improvement += 1
        print(f'No improvement for {epochs_without_improvement} epoch(s).')

    if epochs_without_improvement >= early_stopping_patience:
        print('Early stopping triggered.')
        break

# ==========================
#      MODEL EXPORT
# ==========================
print("\nTraining Complete. Best weights are already saved to encoder_weights.pth and decoder_weights.pth.")
print(f"Best validation loss: {best_val_loss:.4f}")
print("✅ Weights and vocabulary saved successfully!")

Starting Training Engine...

=== Epoch 1/20 ===


Training: 100%|██████████| 2405/2405 [22:49<00:00,  1.76it/s, loss=3.01]


Epoch 1 Completed | Train Loss: 3.5989 | Validation Loss: 3.0750 | Decoder LR: 0.000300
Validation improved. Best checkpoint saved.

=== Epoch 2/20 ===


Training:  54%|█████▍    | 1302/2405 [12:06<10:15,  1.79it/s, loss=2.71]


KeyboardInterrupt: 